# PLATINUM Full Pipeline: 10% Daily-Stratified LightGBM

**Source:** `PROZORRO.PLATINUM.PLATINUM_COMPETITIVE_TENDERS` (4.54M truly-open competitive tenders)

**Target:** `SIGNAL_IS_SINGLE_BIDDER` (binary)

**Regime boundary:** 12 October 2022 (Cabinet Resolution No. 1178 effective date)

**Key changes from prior pipeline:**
- Sources from PLATINUM (not GOLD/DETECTION)
- Excludes defense, framework, competitive dialogue methods
- Drops `FLAG_RESTRICTED_PROCEDURE`, `FLAG_NO_CALL_FOR_TENDER` (structural, not behavioral)
- Removes post-bid and post-award columns (leakage prevention)
- Regime split moved to 2022-10-12 (formalization date, not invasion date)
- Full metrics suite: ROC-AUC, PR-AUC, Brier score, calibration, DeLong CIs
- **Phase 4.5: comprehensive visualization suite** (14 chart types)
- Checkpointing at every phase

---
## Phase 0: Setup & Configuration

In [ ]:
import json
import time
import hashlib
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, log_loss,
    precision_recall_curve, roc_curve, f1_score, precision_score,
    recall_score, accuracy_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, classification_report,
)
from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl
mpl.use('Agg')

warnings.filterwarnings('ignore', category=UserWarning)

# ============================================================
# Plot styling — consistent across the entire notebook
# ============================================================
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 180,
    'savefig.bbox': 'tight',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#E5E5E5',
    'grid.linewidth': 0.6,
    'grid.alpha': 0.7,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'legend.frameon': False,
    'legend.fontsize': 9,
    'xtick.color': '#333333',
    'ytick.color': '#333333',
})

# Variant color palette — semantic, not random.
# Peacetime = blue (calm), Wartime = red (urgency), Pooled = purple (mixed)
VARIANT_COLORS = {
    'peacetime': '#1f77b4',
    'wartime':   '#d62728',
    'pooled':    '#7e3ff2',
}
VARIANT_LIGHT = {
    'peacetime': '#c6dbef',
    'wartime':   '#fcbba1',
    'pooled':    '#dcc8ff',
}

RUN_ID = datetime.utcnow().strftime('V_%Y%m%d_%H%M%S')
RANDOM_SEED = 20260513
SAMPLE_RATE = 0.10

CHECKPOINT_SCHEMA = 'PROZORRO.PLATINUM'
CHECKPOINT_TABLE = f'{CHECKPOINT_SCHEMA}.R_PIPELINE_CHECKPOINTS'

# Local figure directory — figures are saved here, then optionally uploaded to a Snowflake stage
FIG_DIR = Path(f'/tmp/platinum_figs/{RUN_ID}')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Run ID: {RUN_ID}')
print(f'Seed: {RANDOM_SEED}')
print(f'Sample rate: {SAMPLE_RATE}')
print(f'Figure dir: {FIG_DIR}')

In [ ]:
def checkpoint(session, phase: str, status: str, metadata: dict = None):
    meta_json = json.dumps(metadata or {}).replace("'", "''")
    session.sql(f"""
        INSERT INTO {CHECKPOINT_TABLE} (RUN_ID, PHASE, STATUS, METADATA, CREATED_AT)
        SELECT '{RUN_ID}', '{phase}', '{status}', PARSE_JSON('{meta_json}'), CURRENT_TIMESTAMP()
    """).collect()
    print(f'  [CHECKPOINT] {phase} -> {status}')


def savefig(fig, name: str, also_pdf: bool = True):
    """Save a figure as PNG (and optionally PDF) into the run's figure directory."""
    png_path = FIG_DIR / f'{name}.png'
    fig.savefig(png_path)
    if also_pdf:
        fig.savefig(FIG_DIR / f'{name}.pdf')
    plt.close(fig)
    print(f'    saved {png_path.name}')
    return png_path

In [ ]:
%%sql -r ckpt_table_result
CREATE TABLE IF NOT EXISTS PROZORRO.PLATINUM.R_PIPELINE_CHECKPOINTS (
    RUN_ID VARCHAR,
    PHASE VARCHAR,
    STATUS VARCHAR,
    METADATA VARIANT,
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)

---
## Phase 1: 10% Daily-Stratified Sampling from PLATINUM

Stratify by `DATE_CREATED::DATE` to ensure temporal representativeness.
Regime split at **2022-10-12** (Cabinet Resolution No. 1178).

In [ ]:
%%sql -r sample_result
CREATE OR REPLACE TABLE PROZORRO.PLATINUM.SAMPLE_10PCT AS
WITH daily_numbered AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY DATE_CREATED::DATE ORDER BY RANDOM(20260513)) AS rn,
        COUNT(*) OVER (PARTITION BY DATE_CREATED::DATE) AS day_count
    FROM PROZORRO.PLATINUM.PLATINUM_COMPETITIVE_TENDERS
    WHERE DATE_CREATED >= '2016-01-01'
)
SELECT * EXCLUDE (rn, day_count)
FROM daily_numbered
WHERE rn <= GREATEST(1, ROUND(day_count * 0.10))

In [ ]:
%%sql -r sample_stats
SELECT
    _REGIME,
    COUNT(*) AS n,
    ROUND(AVG(CASE WHEN SIGNAL_IS_SINGLE_BIDDER THEN 1 ELSE 0 END), 4) AS base_rate,
    MIN(DATE_CREATED) AS earliest,
    MAX(DATE_CREATED) AS latest
FROM PROZORRO.PLATINUM.SAMPLE_10PCT
GROUP BY _REGIME
ORDER BY _REGIME

---
## Phase 2: Feature Selection & Temporal Splits

Pre-tender features only. No post-bid, post-award, or post-contract columns.

Temporal splits:
- **Peacetime**: train 2016-01-01 to 2021-06-30 | val 2021-07-01 to 2022-03-31 | test 2022-04-01 to 2022-10-11
- **Wartime**: train 2022-10-12 to 2024-12-31 | val 2025-01-01 to 2025-06-30 | test 2025-07-01 onward
- **Pooled**: uses both with `_IS_WARTIME` as feature

In [ ]:
PRE_TENDER_FEATURES = [
    'PROCUREMENT_METHOD',
    'PROCUREMENT_METHOD_TYPE',
    'FLAG_BELOW_THRESHOLD_WITH_BIDDING',
    'FLAG_NON_PRICE_AWARD_CRITERIA',
    'SIGNAL_ENQUIRY_PERIOD_HOURS',
    'SIGNAL_TENDER_PERIOD_HOURS',
    'SIGNAL_ENQUIRY_PERIOD_BELOW_LEGAL_MIN',
    'SIGNAL_TENDER_PERIOD_BELOW_LEGAL_MIN',
    'SIGNAL_TENDER_PERIOD_RATIO_TO_PEER_MEDIAN',
    'FLAG_IS_DECEMBER_PUBLISH',
    'FLAG_IS_YEAR_BOUNDARY_PUBLISH',
    'FLAG_IS_UKRAINIAN_HOLIDAY_PUBLISH',
    'FLAG_IS_WARTIME_REGIME',
    'FLAG_IS_WARTIME_SIMPLIFIED',
    'VALUE_AMOUNT',
    'SIGNAL_NEAR_THRESHOLD_RATIO',
    'FLAG_NEAR_THRESHOLD_CLUSTER',
    'SIGNAL_ITEM_COUNT',
    'SIGNAL_CPV_HETEROGENEITY',
    'SIGNAL_LARGE_LOT_COUNT',
    'STAT_PRIMARY_CPV_4DIGIT',
    'FLAG_MISSING_TENDER_DOCUMENTATION',
    'FLAG_DESCRIPTION_LENGTH_SUSPICIOUS',
    'FLAG_CPV_CONSTRUCTION',
    'FLAG_CPV_MEDICAL_PHARMA',
    'FLAG_CPV_IT_ELECTRONICS',
    'FLAG_CPV_ENGINEERING_SERVICES',
    'FLAG_CPV_ENERGY',
    'FLAG_CPV_FOODSERVICE',
    'FLAG_RECONSTRUCTION_RELATED',
    'PROCURER_KIND',
    'PROCURER_REGION',
    'FLAG_PROCURER_DEFENSE',
    'FLAG_PROCURER_MUNICIPAL',
    'COMPOSITE_CRI_PROCEDURE_RESTRICTIVENESS',
    '_PEER_GROUP_SIZE',
]

TARGET = 'SIGNAL_IS_SINGLE_BIDDER'
ID_COL = 'TENDER_HASH_ID'
DATE_COL = 'DATE_CREATED'
REGIME_COL = '_REGIME'

CATEGORICAL_FEATURES = [
    'PROCUREMENT_METHOD',
    'PROCUREMENT_METHOD_TYPE',
    'STAT_PRIMARY_CPV_4DIGIT',
    'PROCURER_KIND',
    'PROCURER_REGION',
]

REGIME_BOUNDARY = '2022-10-12'

SPLITS = {
    'peacetime': {
        'train': ('2016-01-01', '2021-06-30'),
        'val':   ('2021-07-01', '2022-03-31'),
        'test':  ('2022-04-01', '2022-10-11'),
    },
    'wartime': {
        'train': ('2022-10-12', '2024-12-31'),
        'val':   ('2025-01-01', '2025-06-30'),
        'test':  ('2025-07-01', '2026-12-31'),
    },
}

print(f'Features: {len(PRE_TENDER_FEATURES)}')
print(f'Categoricals: {len(CATEGORICAL_FEATURES)}')
print(f'Regime boundary: {REGIME_BOUNDARY}')

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

sample_df = session.table('PROZORRO.PLATINUM.SAMPLE_10PCT').to_pandas()
print(f'Sample loaded: {len(sample_df):,} rows')
print(f'Columns: {sample_df.shape[1]}')
print(f'Regime distribution:')
print(sample_df[REGIME_COL].value_counts())

In [ ]:
y = sample_df[TARGET].astype(int)
X = sample_df[PRE_TENDER_FEATURES].copy()
dates = pd.to_datetime(sample_df[DATE_COL])
regime = sample_df[REGIME_COL]
tender_ids = sample_df[ID_COL]

all_null_cols = X.columns[X.isnull().all()].tolist()
if all_null_cols:
    print(f'Dropping {len(all_null_cols)} all-null columns: {all_null_cols}')
    X = X.drop(columns=all_null_cols)
    PRE_TENDER_FEATURES = [f for f in PRE_TENDER_FEATURES if f not in all_null_cols]

for col in CATEGORICAL_FEATURES:
    if col in X.columns:
        X[col] = X[col].astype('category')

for col in X.select_dtypes(include=['bool']).columns:
    X[col] = X[col].astype(int)

for col in X.select_dtypes(include=['object']).columns:
    if col not in CATEGORICAL_FEATURES:
        X[col] = pd.to_numeric(X[col], errors='coerce')

# Cache for null-rate chart later
feature_null_rates = X.isnull().mean().sort_values(ascending=False)

print(f'X shape: {X.shape}')
print(f'Target base rate: {y.mean():.4f}')
print(f'Nulls per col (top 10):')
print(feature_null_rates.head(10))

In [ ]:
def make_split_mask(dates_series, start, end):
    return (dates_series >= start) & (dates_series <= end)

datasets = {}
for variant, boundaries in SPLITS.items():
    ds = {}
    for split_name, (start, end) in boundaries.items():
        mask = make_split_mask(dates, start, end)
        ds[split_name] = {
            'X': X[mask].copy(),
            'y': y[mask].copy(),
            'ids': tender_ids[mask].copy(),
            'dates': dates[mask].copy(),
        }
        print(f'  {variant}/{split_name}: {mask.sum():,} rows, base_rate={y[mask].mean():.4f}')
    datasets[variant] = ds

pooled_train_mask = make_split_mask(dates, '2016-01-01', '2024-12-31')
pooled_val_mask = make_split_mask(dates, '2025-01-01', '2025-06-30')
pooled_test_peace_mask = make_split_mask(dates, '2022-04-01', '2022-10-11')
pooled_test_war_mask = make_split_mask(dates, '2025-07-01', '2026-12-31')

X_pooled = X.copy()
X_pooled['_IS_WARTIME'] = (regime == 'wartime').astype(int)

datasets['pooled'] = {
    'train': {'X': X_pooled[pooled_train_mask], 'y': y[pooled_train_mask], 'ids': tender_ids[pooled_train_mask], 'dates': dates[pooled_train_mask]},
    'val': {'X': X_pooled[pooled_val_mask], 'y': y[pooled_val_mask], 'ids': tender_ids[pooled_val_mask], 'dates': dates[pooled_val_mask]},
    'test_wartime': {'X': X_pooled[pooled_test_war_mask], 'y': y[pooled_test_war_mask], 'ids': tender_ids[pooled_test_war_mask], 'dates': dates[pooled_test_war_mask]},
    'test_peacetime': {'X': X_pooled[pooled_test_peace_mask], 'y': y[pooled_test_peace_mask], 'ids': tender_ids[pooled_test_peace_mask], 'dates': dates[pooled_test_peace_mask]},
}
print(f'\n  pooled/train: {pooled_train_mask.sum():,}')
print(f'  pooled/val: {pooled_val_mask.sum():,}')
print(f'  pooled/test_wartime: {pooled_test_war_mask.sum():,}')
print(f'  pooled/test_peacetime: {pooled_test_peace_mask.sum():,}')

checkpoint(session, 'phase2_splits', 'complete', {
    'peacetime_train': int(datasets['peacetime']['train']['y'].shape[0]),
    'wartime_train': int(datasets['wartime']['train']['y'].shape[0]),
    'n_features': X.shape[1],
})

### 2.1 Sample sanity charts (built before training so leakage in splits is visually obvious)

In [ ]:
# Chart 1 — Temporal representativeness of the daily-stratified sample.
# Two-panel: monthly sample counts (top) and monthly single-bidder base rate (bottom),
# with regime boundary and split boundaries marked.

monthly = pd.DataFrame({'date': dates, 'y': y, 'regime': regime})
monthly['month'] = monthly['date'].dt.to_period('M').dt.to_timestamp()
agg = monthly.groupby('month').agg(n=('y', 'size'), base_rate=('y', 'mean')).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True,
                         gridspec_kw={'height_ratios': [1, 1.2], 'hspace': 0.18})

ax = axes[0]
ax.fill_between(agg['month'], agg['n'], color='#4a6fa5', alpha=0.55, linewidth=0)
ax.plot(agg['month'], agg['n'], color='#1e3a5f', linewidth=1.2)
ax.set_ylabel('Tenders in sample')
ax.set_title('Daily-stratified 10% sample: temporal coverage')
ax.axvline(pd.Timestamp(REGIME_BOUNDARY), color='#d62728', linestyle='--', linewidth=1.3, alpha=0.8)
ax.text(pd.Timestamp(REGIME_BOUNDARY), ax.get_ylim()[1] * 0.92, '  Resolution 1178',
        color='#d62728', fontsize=9, fontweight='bold')

ax = axes[1]
ax.plot(agg['month'], agg['base_rate'], color='#c92a2a', linewidth=1.6, marker='o', markersize=3.2)
ax.set_ylabel('Single-bidder rate')
ax.set_xlabel('Month')
ax.set_title('Monthly single-bidder base rate (target = 1 share)')
ax.axvline(pd.Timestamp(REGIME_BOUNDARY), color='#d62728', linestyle='--', linewidth=1.3, alpha=0.8)
ax.axhline(y.mean(), color='#888', linestyle=':', linewidth=1, label=f'Overall mean = {y.mean():.3f}')
ax.legend(loc='upper left')

# Shade split regions on the bottom panel
for variant, boundaries in SPLITS.items():
    for split_name, (start, end) in boundaries.items():
        color = {'train': '#2ecc71', 'val': '#f39c12', 'test': '#e74c3c'}[split_name]
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.06, color=color, linewidth=0)

legend_patches = [mpatches.Patch(color=c, alpha=0.18, label=lbl)
                  for lbl, c in [('train', '#2ecc71'), ('val', '#f39c12'), ('test', '#e74c3c')]]
ax.legend(handles=[mpl.lines.Line2D([], [], color='#888', linestyle=':', label=f'Overall mean = {y.mean():.3f}')] + legend_patches,
          loc='upper left', ncol=4)

savefig(fig, '01_sample_temporal_coverage')

In [ ]:
# Chart 2 — Top-20 features by missingness, split by regime so we can see whether
# any feature's availability changed materially after the regime boundary.
# Compute null rates per regime without groupby.apply (avoids pandas include_groups warnings).
regime_aligned = pd.Series(regime.values, index=X.index)
null_by_regime = pd.DataFrame({
    r: X[regime_aligned == r].isnull().mean()
    for r in regime_aligned.unique()
})
null_by_regime['_max'] = null_by_regime.max(axis=1)
null_by_regime = null_by_regime.sort_values('_max', ascending=False).drop(columns='_max').head(20)

fig, ax = plt.subplots(figsize=(10, 7))
idx = np.arange(len(null_by_regime))
width = 0.4
ax.barh(idx - width/2, null_by_regime.get('peacetime', pd.Series(0, index=null_by_regime.index)),
        width, color=VARIANT_COLORS['peacetime'], label='peacetime')
ax.barh(idx + width/2, null_by_regime.get('wartime',  pd.Series(0, index=null_by_regime.index)),
        width, color=VARIANT_COLORS['wartime'], label='wartime')
ax.set_yticks(idx)
ax.set_yticklabels(null_by_regime.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Null rate')
ax.set_title('Top-20 features by missingness — peacetime vs wartime')
ax.legend(loc='lower right')
ax.set_xlim(0, max(0.05, null_by_regime.values.max() * 1.1))
savefig(fig, '02_feature_null_rates')

---
## Phase 3: Train LightGBM Models

Three variants: peacetime, wartime, pooled.
Hyperparameters from prior random-search (notebook 13). Capture the full evals dictionary so we can plot learning curves later.

In [ ]:
LGB_PARAMS = {
    'objective': 'binary',
    'metric': ['binary_logloss', 'auc'],   # track BOTH so we can plot AUC learning curves too
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 63,
    'max_depth': 7,
    'min_child_samples': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': RANDOM_SEED,
    'verbose': -1,
    'n_jobs': -1,
    'is_unbalance': False,
}

NUM_BOOST_ROUND = 1500
EARLY_STOPPING_ROUNDS = 50

print('LightGBM params:')
for k, v in LGB_PARAMS.items():
    print(f'  {k}: {v}')

In [ ]:
models = {}
training_logs = {}
evals_history = {}   # variant -> {dataset_name: {metric_name: [values per round]}}

for variant in ['peacetime', 'wartime', 'pooled']:
    print(f'\n{"="*60}')
    print(f'Training: {variant.upper()}')
    print(f'{"="*60}')
    
    ds = datasets[variant]
    X_tr = ds['train']['X']
    y_tr = ds['train']['y']
    X_val = ds['val']['X']
    y_val = ds['val']['y']
    
    cat_cols = [c for c in CATEGORICAL_FEATURES if c in X_tr.columns]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols, free_raw_data=False)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, free_raw_data=False, reference=dtrain)
    
    evals_result = {}
    callbacks = [
        lgb.early_stopping(EARLY_STOPPING_ROUNDS),
        lgb.log_evaluation(100),
        lgb.record_evaluation(evals_result),
    ]
    
    t0 = time.time()
    bst = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'val'],
        callbacks=callbacks,
    )
    elapsed = time.time() - t0
    
    models[variant] = bst
    evals_history[variant] = evals_result
    training_logs[variant] = {
        'best_iteration': bst.best_iteration,
        'best_score': {k: dict(v) for k, v in bst.best_score.items()},
        'elapsed_sec': round(elapsed, 1),
        'n_features': X_tr.shape[1],
        'n_train': len(y_tr),
        'n_val': len(y_val),
    }
    
    val_pred = bst.predict(X_val, num_iteration=bst.best_iteration)
    val_auc = roc_auc_score(y_val, val_pred)
    val_logloss = log_loss(y_val, val_pred)
    
    print(f'\n  Best iteration: {bst.best_iteration}')
    print(f'  Val AUC: {val_auc:.4f}')
    print(f'  Val LogLoss: {val_logloss:.4f}')
    print(f'  Time: {elapsed:.1f}s')
    
    training_logs[variant]['val_auc'] = val_auc
    training_logs[variant]['val_logloss'] = val_logloss

checkpoint(session, 'phase3_training', 'complete', {
    k: {'val_auc': v['val_auc'], 'best_iter': v['best_iteration']}
    for k, v in training_logs.items()
})

In [ ]:
# Chart 3 — Learning curves: train vs val logloss AND AUC across boosting rounds,
# one row per variant. Best-iteration marker is drawn as a vertical dashed line.

fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=False)

for row, variant in enumerate(['peacetime', 'wartime', 'pooled']):
    hist = evals_history[variant]
    best_iter = training_logs[variant]['best_iteration']
    color = VARIANT_COLORS[variant]
    
    # --- Logloss panel ---
    ax = axes[row, 0]
    rounds = np.arange(1, len(hist['train']['binary_logloss']) + 1)
    ax.plot(rounds, hist['train']['binary_logloss'], color=color, linewidth=1.4, label='train')
    ax.plot(rounds, hist['val']['binary_logloss'],   color=color, linewidth=1.4,
            linestyle='--', alpha=0.85, label='val')
    ax.axvline(best_iter, color='#444', linestyle=':', linewidth=1, label=f'best iter = {best_iter}')
    ax.set_title(f'{variant.upper()} — log-loss')
    ax.set_xlabel('Boosting round')
    ax.set_ylabel('Binary log-loss')
    ax.legend(loc='upper right')
    
    # --- AUC panel ---
    ax = axes[row, 1]
    ax.plot(rounds, hist['train']['auc'], color=color, linewidth=1.4, label='train')
    ax.plot(rounds, hist['val']['auc'],   color=color, linewidth=1.4,
            linestyle='--', alpha=0.85, label='val')
    ax.axvline(best_iter, color='#444', linestyle=':', linewidth=1)
    ax.set_title(f'{variant.upper()} — AUC')
    ax.set_xlabel('Boosting round')
    ax.set_ylabel('ROC-AUC')
    ax.legend(loc='lower right')

fig.suptitle('Learning curves (train vs val) — all three variants', y=1.00, fontsize=13, fontweight='bold')
fig.tight_layout()
savefig(fig, '03_learning_curves')

---
## Phase 4: Evaluation & SHAP

Comprehensive metrics on test sets + cross-regime transfer + SHAP importance. All predictions are cached for the visualization phase that follows.

In [ ]:
def evaluate_model(bst, X_test, y_test, label):
    pred = bst.predict(X_test, num_iteration=bst.best_iteration)
    pred_class = (pred >= 0.5).astype(int)
    metrics = {
        'label': label,
        'n': len(y_test),
        'base_rate': float(y_test.mean()),
        'auc_roc': float(roc_auc_score(y_test, pred)),
        'auc_pr': float(average_precision_score(y_test, pred)),
        'brier': float(brier_score_loss(y_test, pred)),
        'logloss': float(log_loss(y_test, pred)),
        'f1': float(f1_score(y_test, pred_class)),
        'precision': float(precision_score(y_test, pred_class, zero_division=0)),
        'recall': float(recall_score(y_test, pred_class, zero_division=0)),
        'accuracy': float(accuracy_score(y_test, pred_class)),
        'mcc': float(matthews_corrcoef(y_test, pred_class)),
        'kappa': float(cohen_kappa_score(y_test, pred_class)),
        'pred_mean': float(pred.mean()),
        'pred_std': float(pred.std()),
    }
    return metrics, pred

all_results = []
all_predictions = {}   # label -> {'y_true': ..., 'y_score': ..., 'variant': ..., 'split': ..., 'kind': ...}

for variant in ['peacetime', 'wartime', 'pooled']:
    bst = models[variant]
    ds = datasets[variant]
    test_splits = {k: v for k, v in ds.items() if 'test' in k}
    for split_name, split_data in test_splits.items():
        label = f'{variant}/{split_name}'
        metrics, pred = evaluate_model(bst, split_data['X'], split_data['y'], label)
        metrics['variant'] = variant
        metrics['split'] = split_name
        metrics['kind'] = 'native'
        all_results.append(metrics)
        all_predictions[label] = {
            'y_true': split_data['y'].values, 'y_score': pred,
            'variant': variant, 'split': split_name, 'kind': 'native',
        }
        print(f'{label}: AUC={metrics["auc_roc"]:.4f}, LogLoss={metrics["logloss"]:.4f}, n={metrics["n"]:,}')

print('\n--- Cross-regime transfer ---')
transfers = [
    ('peacetime', 'wartime', 'test'),
    ('wartime', 'peacetime', 'test'),
]
for model_variant, data_variant, split_name in transfers:
    bst = models[model_variant]
    data = datasets[data_variant][split_name]
    X_t = data['X']
    if model_variant == 'pooled' and '_IS_WARTIME' not in X_t.columns:
        X_t = X_t.copy()
        X_t['_IS_WARTIME'] = int(data_variant == 'wartime')
    label = f'{model_variant}_on_{data_variant}/{split_name}'
    metrics, pred = evaluate_model(bst, X_t, data['y'], label)
    metrics['variant'] = model_variant
    metrics['split'] = f'transfer_{data_variant}_{split_name}'
    metrics['kind'] = 'transfer'
    all_results.append(metrics)
    all_predictions[label] = {
        'y_true': data['y'].values, 'y_score': pred,
        'variant': model_variant, 'split': metrics['split'], 'kind': 'transfer',
    }
    print(f'{label}: AUC={metrics["auc_roc"]:.4f}')

results_df = pd.DataFrame(all_results)
print(f'\nTotal evaluation runs: {len(results_df)}')
results_df[['label', 'auc_roc', 'auc_pr', 'logloss', 'brier', 'n', 'base_rate']]

In [ ]:
import shap

shap_results = {}
SHAP_SAMPLE_SIZE = 10000

# Explicit mapping — for pooled, we want the WARTIME test set, not whichever happens to be last in dict order.
SHAP_TEST_SPLIT = {
    'peacetime': 'test',
    'wartime':   'test',
    'pooled':    'test_wartime',
}

for variant in ['peacetime', 'wartime', 'pooled']:
    bst = models[variant]
    ds = datasets[variant]
    X_test = ds[SHAP_TEST_SPLIT[variant]]['X']
    
    if len(X_test) > SHAP_SAMPLE_SIZE:
        idx = np.random.RandomState(RANDOM_SEED).choice(len(X_test), SHAP_SAMPLE_SIZE, replace=False)
        X_shap = X_test.iloc[idx]
    else:
        X_shap = X_test
    
    explainer = shap.TreeExplainer(bst)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    feature_importance = pd.DataFrame({
        'feature': X_shap.columns,
        'mean_abs_shap': mean_abs_shap,
    }).sort_values('mean_abs_shap', ascending=False)
    
    base_value = explainer.expected_value
    if isinstance(base_value, (list, np.ndarray)):
        base_value = base_value[1] if len(np.atleast_1d(base_value)) > 1 else float(np.atleast_1d(base_value)[0])
    
    shap_results[variant] = {
        'importance': feature_importance,
        'base_value': float(base_value),
        'shap_values': shap_values,         # kept for beeswarm
        'X_shap': X_shap,                   # kept for beeswarm
    }
    
    print(f'\n{variant.upper()} — Top 15 features (base_value={shap_results[variant]["base_value"]:.4f}):')
    print(feature_importance.head(15).to_string(index=False))

checkpoint(session, 'phase4_evaluation', 'complete', {
    'n_results': len(all_results),
    'wartime_test_auc': float(results_df[results_df['label'] == 'wartime/test']['auc_roc'].iloc[0])
        if (results_df['label'] == 'wartime/test').any() else None,
})

---
## Phase 4.5: Comprehensive Visualization Suite

Fourteen production charts covering: model discrimination (ROC, PR), calibration, confusion matrices, threshold sensitivity, classification reports, learning curves (above), feature importance comparison & SHAP beeswarm, cross-regime transfer, score distributions, and a paper-ready summary panel.

All figures are saved as PNG + PDF to the run's figure directory and (optionally) uploaded to a Snowflake stage in Phase 5.

In [ ]:
def delong_ci(y_true, y_score, alpha=0.05):
    """DeLong (1988) variance estimator for the AUC, returning (auc, lo, hi).
    Implementation follows Sun & Xu (2014). Used in the ROC legend."""
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    pos = y_score[y_true == 1]
    neg = y_score[y_true == 0]
    m, n = len(pos), len(neg)
    if m == 0 or n == 0:
        return float('nan'), float('nan'), float('nan')
    # Mid-ranks for ties (compact V-statistic form).
    def midrank(x):
        order = np.argsort(x, kind='mergesort')
        ranks = np.empty(len(x))
        x_sorted = x[order]
        i = 0
        while i < len(x):
            j = i
            while j < len(x) - 1 and x_sorted[j + 1] == x_sorted[i]:
                j += 1
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1
            i = j + 1
        return ranks
    all_scores = np.concatenate([pos, neg])
    all_labels = np.concatenate([np.ones(m), np.zeros(n)])
    r_all = midrank(all_scores)
    r_pos_only = midrank(pos)
    r_neg_only = midrank(neg)
    V10 = (r_all[:m] - r_pos_only) / n
    V01 = 1.0 - (r_all[m:] - r_neg_only) / m
    auc = V10.mean()
    s10 = V10.var(ddof=1) / m
    s01 = V01.var(ddof=1) / n
    se = np.sqrt(s10 + s01)
    from scipy.stats import norm
    z = norm.ppf(1 - alpha / 2)
    return float(auc), float(max(0.0, auc - z * se)), float(min(1.0, auc + z * se))


def pretty_label(label: str) -> str:
    return label.replace('_', ' ').replace('/', ' → ')

In [ ]:
# All native test sets on one ROC axes, plus the two cross-regime transfers as dashed lines.
# This is the single most informative chart for comparing regime discriminability.

fig, ax = plt.subplots(figsize=(7.5, 7.5))

order = [
    ('peacetime/test',        'solid'),
    ('wartime/test',          'solid'),
    ('pooled/test_peacetime', 'solid'),
    ('pooled/test_wartime',   'solid'),
    ('peacetime_on_wartime/test', 'dashed'),
    ('wartime_on_peacetime/test', 'dashed'),
]

for label, ls in order:
    if label not in all_predictions:
        continue
    d = all_predictions[label]
    fpr, tpr, _ = roc_curve(d['y_true'], d['y_score'])
    auc, lo, hi = delong_ci(d['y_true'], d['y_score'])
    color = VARIANT_COLORS[d['variant']]
    alpha = 1.0 if ls == 'solid' else 0.6
    ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=2 if ls == 'solid' else 1.5,
            alpha=alpha, label=f'{pretty_label(label)}  AUC = {auc:.3f} [{lo:.3f}, {hi:.3f}]')

ax.plot([0, 1], [0, 1], color='#888', linestyle=':', linewidth=1, label='chance')
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curves — native test sets and cross-regime transfer\n(95% DeLong CI in legend)')
ax.legend(loc='lower right', fontsize=8.5)
ax.set_aspect('equal')
savefig(fig, '04_roc_overlay')

In [ ]:
# PR is the honest metric for imbalanced data. The baseline (dotted line per curve)
# is the test-set base rate — i.e. what 'guessing the prior' gets you.

fig, ax = plt.subplots(figsize=(7.5, 7.5))

for label, ls in order:
    if label not in all_predictions:
        continue
    d = all_predictions[label]
    p, r, _ = precision_recall_curve(d['y_true'], d['y_score'])
    ap = average_precision_score(d['y_true'], d['y_score'])
    base = d['y_true'].mean()
    color = VARIANT_COLORS[d['variant']]
    alpha = 1.0 if ls == 'solid' else 0.6
    ax.plot(r, p, color=color, linestyle=ls, linewidth=2 if ls == 'solid' else 1.5,
            alpha=alpha, label=f'{pretty_label(label)}  AP = {ap:.3f}  (base = {base:.3f})')

ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall curves\n(baseline = test-set base rate)')
ax.legend(loc='lower left', fontsize=8.5)
ax.set_aspect('equal')
savefig(fig, '05_pr_overlay')

In [ ]:
# For each native test set: 10-bin reliability curve on top, predicted-probability
# histogram below. A well-calibrated model hugs the diagonal.

native_labels = ['peacetime/test', 'wartime/test', 'pooled/test_peacetime', 'pooled/test_wartime']
native_labels = [l for l in native_labels if l in all_predictions]

fig = plt.figure(figsize=(15, 8))
gs = GridSpec(2, len(native_labels), height_ratios=[2.4, 1], hspace=0.12, wspace=0.25)

for i, label in enumerate(native_labels):
    d = all_predictions[label]
    color = VARIANT_COLORS[d['variant']]
    frac_pos, mean_pred = calibration_curve(d['y_true'], d['y_score'], n_bins=10, strategy='quantile')
    brier = brier_score_loss(d['y_true'], d['y_score'])

    ax = fig.add_subplot(gs[0, i])
    ax.plot([0, 1], [0, 1], color='#999', linestyle=':', linewidth=1)
    ax.plot(mean_pred, frac_pos, color=color, marker='o', linewidth=2, markersize=6)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Empirical positive rate')
    ax.set_title(f'{pretty_label(label)}\nBrier = {brier:.4f}')
    ax.set_aspect('equal')

    ax2 = fig.add_subplot(gs[1, i])
    ax2.hist(d['y_score'][d['y_true'] == 0], bins=40, color='#888', alpha=0.6, label='y = 0', density=True)
    ax2.hist(d['y_score'][d['y_true'] == 1], bins=40, color=color, alpha=0.7, label='y = 1', density=True)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('Predicted probability')
    ax2.set_ylabel('Density')
    if i == 0:
        ax2.legend(loc='upper center', fontsize=8)

fig.suptitle('Calibration (top) and predicted-probability histograms by true class (bottom)',
             y=1.02, fontsize=13, fontweight='bold')
savefig(fig, '06_calibration')

In [ ]:
# Row-normalized so that the diagonal reads as recall-per-class and class imbalance
# doesn't drown out the minority class. Threshold = 0.5 (note: NOT necessarily optimal).

fig, axes = plt.subplots(1, len(native_labels), figsize=(4.2 * len(native_labels), 4.5))
if len(native_labels) == 1:
    axes = [axes]

cmap = LinearSegmentedColormap.from_list('cm', ['#f7fbff', '#08306b'])

for ax, label in zip(axes, native_labels):
    d = all_predictions[label]
    pred_class = (d['y_score'] >= 0.5).astype(int)
    cm = confusion_matrix(d['y_true'], pred_class)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            txt_color = 'white' if cm_norm[i, j] > 0.55 else '#222'
            ax.text(j, i, f'{cm_norm[i, j]:.2%}\n(n={cm[i, j]:,})',
                    ha='center', va='center', color=txt_color, fontsize=10, fontweight='bold')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['pred 0', 'pred 1'])
    ax.set_yticklabels(['true 0', 'true 1'])
    ax.set_title(pretty_label(label), fontsize=11)
    ax.grid(False)

fig.colorbar(im, ax=axes, fraction=0.022, pad=0.02, label='Row-normalized rate')
fig.suptitle('Confusion matrices @ threshold = 0.5 (row-normalized)',
             y=1.05, fontsize=13, fontweight='bold')
savefig(fig, '07_confusion_matrices')

In [ ]:
# precision/recall/F1 per class, rendered as a heatmap for the four native test sets.

rows = []
for label in native_labels:
    d = all_predictions[label]
    pred_class = (d['y_score'] >= 0.5).astype(int)
    rep = classification_report(d['y_true'], pred_class, output_dict=True, zero_division=0)
    for cls in ['0', '1', 'macro avg', 'weighted avg']:
        if cls in rep:
            rows.append({
                'label': pretty_label(label),
                'class': cls,
                'precision': rep[cls]['precision'],
                'recall':    rep[cls]['recall'],
                'f1':        rep[cls]['f1-score'],
            })
report_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
metric_cmaps = {'precision': 'Blues', 'recall': 'Greens', 'f1': 'Purples'}
for ax, metric in zip(axes, ['precision', 'recall', 'f1']):
    pivot = report_df.pivot(index='label', columns='class', values=metric)
    pivot = pivot[['0', '1', 'macro avg', 'weighted avg']]
    im = ax.imshow(pivot.values, cmap=metric_cmaps[metric], vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title(metric.upper(), fontweight='bold')
    ax.grid(False)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v > 0.55 else '#222', fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)

fig.suptitle('Classification report heatmap — per-class metrics @ threshold = 0.5',
             y=1.02, fontsize=13, fontweight='bold')
savefig(fig, '08_classification_report_heatmap')

In [ ]:
# Precision, recall, F1, MCC, and accuracy as a function of decision threshold.
# One subplot per native test set. The 0.5 default is rarely optimal for imbalanced targets.

fig, axes = plt.subplots(1, len(native_labels), figsize=(4.5 * len(native_labels), 4.5), sharey=True)
if len(native_labels) == 1:
    axes = [axes]

thresholds = np.linspace(0.01, 0.99, 60)
for ax, label in zip(axes, native_labels):
    d = all_predictions[label]
    P, R, F1, MCC, ACC = [], [], [], [], []
    for t in thresholds:
        pc = (d['y_score'] >= t).astype(int)
        P.append(precision_score(d['y_true'], pc, zero_division=0))
        R.append(recall_score(d['y_true'], pc, zero_division=0))
        F1.append(f1_score(d['y_true'], pc, zero_division=0))
        MCC.append(matthews_corrcoef(d['y_true'], pc) if pc.sum() > 0 else 0.0)
        ACC.append(accuracy_score(d['y_true'], pc))
    
    ax.plot(thresholds, P,  color='#1f77b4', label='precision', linewidth=1.6)
    ax.plot(thresholds, R,  color='#2ca02c', label='recall',    linewidth=1.6)
    ax.plot(thresholds, F1, color='#d62728', label='F1',        linewidth=1.8)
    ax.plot(thresholds, MCC,color='#9467bd', label='MCC',       linewidth=1.4, linestyle='--')
    ax.plot(thresholds, ACC,color='#888',    label='accuracy',  linewidth=1.2, linestyle=':')
    
    best_f1_t = thresholds[int(np.argmax(F1))]
    ax.axvline(best_f1_t, color='#d62728', linestyle=':', alpha=0.6)
    ax.axvline(0.5, color='#444', linestyle=':', alpha=0.4)
    ax.text(best_f1_t, 0.02, f'  argmax F1 = {best_f1_t:.2f}', color='#d62728', fontsize=8, rotation=90, va='bottom')
    
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Decision threshold')
    ax.set_title(pretty_label(label), fontsize=10)
    if ax is axes[0]:
        ax.set_ylabel('Metric value')
        ax.legend(loc='center right', fontsize=8)

fig.suptitle('Threshold sensitivity — choose the operating point, don\'t assume 0.5',
             y=1.02, fontsize=13, fontweight='bold')
savefig(fig, '09_threshold_sensitivity')

In [ ]:
# The displacement chart. Pivot top features across the three models so that
# any feature whose importance is large in one regime and small in another is
# instantly visible. This is the figure that goes in the paper.

top_n = 20
imp_long = []
for variant in ['peacetime', 'wartime', 'pooled']:
    df = shap_results[variant]['importance'].head(top_n).copy()
    df['variant'] = variant
    imp_long.append(df)
imp_long = pd.concat(imp_long, ignore_index=True)

union_features = (imp_long.groupby('feature')['mean_abs_shap']
                          .max().sort_values(ascending=False).head(30).index.tolist())

imp_wide = (imp_long[imp_long['feature'].isin(union_features)]
            .pivot_table(index='feature', columns='variant', values='mean_abs_shap', fill_value=0))
imp_wide = imp_wide.loc[union_features]
imp_wide = imp_wide[[v for v in ['peacetime', 'wartime', 'pooled'] if v in imp_wide.columns]]

fig, ax = plt.subplots(figsize=(11, 9))
idx = np.arange(len(imp_wide))
bar_h = 0.27
offsets = {'peacetime': -bar_h, 'wartime': 0, 'pooled': bar_h}
for variant in imp_wide.columns:
    ax.barh(idx + offsets[variant], imp_wide[variant].values, height=bar_h,
            color=VARIANT_COLORS[variant], label=variant, alpha=0.92)

ax.set_yticks(idx)
ax.set_yticklabels(imp_wide.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP value|')
ax.set_title(f'Top-{len(union_features)} features by SHAP importance — peacetime vs wartime vs pooled\n'
             f'(union of each variant\'s top {top_n})')
ax.legend(loc='lower right')
savefig(fig, '10_shap_importance_comparison')

In [ ]:
# Direction of effect for top features in the wartime model. Each dot is one tender;
# x-axis is the SHAP value (push toward / away from single-bidder), color is the
# feature's standardized value.

import shap as shap_lib

v = 'wartime'
top_features = shap_results[v]['importance']['feature'].head(20).tolist()
X_shap = shap_results[v]['X_shap'][top_features].copy()
for c in X_shap.columns:
    if isinstance(X_shap[c].dtype, pd.CategoricalDtype):
        X_shap[c] = X_shap[c].cat.codes
shap_vals = shap_results[v]['shap_values'][:, [list(shap_results[v]['X_shap'].columns).index(f) for f in top_features]]

fig = plt.figure(figsize=(10, 9))
shap_lib.summary_plot(
    shap_vals, X_shap, feature_names=top_features,
    plot_type='dot', show=False, max_display=20, color_bar=True,
)
fig = plt.gcf()
fig.suptitle('SHAP beeswarm — WARTIME model on wartime test set\n'
             '(red = high feature value, blue = low)', y=1.02, fontsize=12, fontweight='bold')
savefig(fig, '11_shap_beeswarm_wartime')

In [ ]:
# 3 (model regime) x 2 (test regime) AUC heatmap. Diagonal = native performance,
# off-diagonal = cross-regime transfer. The diagonal-vs-off-diagonal gap quantifies
# regime-specific signal.

transfer_matrix = pd.DataFrame(
    index=['peacetime', 'wartime', 'pooled'],
    columns=['peacetime_test', 'wartime_test'],
    dtype=float,
)
mapping = {
    ('peacetime', 'peacetime_test'): 'peacetime/test',
    ('wartime',   'wartime_test'):   'wartime/test',
    ('pooled',    'peacetime_test'): 'pooled/test_peacetime',
    ('pooled',    'wartime_test'):   'pooled/test_wartime',
    ('peacetime', 'wartime_test'):   'peacetime_on_wartime/test',
    ('wartime',   'peacetime_test'): 'wartime_on_peacetime/test',
}
for (model_v, test_v), key in mapping.items():
    row = results_df[results_df['label'] == key]
    if len(row):
        transfer_matrix.loc[model_v, test_v] = float(row['auc_roc'].iloc[0])

fig, ax = plt.subplots(figsize=(6.5, 6))
cmap = LinearSegmentedColormap.from_list('tr', ['#fff5eb', '#7f2704'])
im = ax.imshow(transfer_matrix.values.astype(float), cmap=cmap, vmin=0.5, vmax=1.0, aspect='auto')
for i in range(transfer_matrix.shape[0]):
    for j in range(transfer_matrix.shape[1]):
        v = transfer_matrix.values[i, j]
        if pd.notnull(v):
            ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                    color='white' if v > 0.78 else '#222',
                    fontsize=13, fontweight='bold')
ax.set_xticks(range(transfer_matrix.shape[1]))
ax.set_xticklabels(transfer_matrix.columns)
ax.set_yticks(range(transfer_matrix.shape[0]))
ax.set_yticklabels(transfer_matrix.index)
ax.set_xlabel('Tested on')
ax.set_ylabel('Trained on')
ax.set_title('Cross-regime transfer — ROC-AUC')
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.045, pad=0.04, label='AUC')
savefig(fig, '12_cross_regime_transfer')

In [ ]:
# Violin-style score distributions per native test set, split by true class. Shows
# whether the model's separation degrades between regimes, and whether the score
# distribution itself shifts (a common pre-cursor to calibration drift).

fig, ax = plt.subplots(figsize=(11, 6))

positions = []
ticklabels = []
for i, label in enumerate(native_labels):
    d = all_predictions[label]
    pos = i * 3
    
    neg_scores = d['y_score'][d['y_true'] == 0]
    pos_scores = d['y_score'][d['y_true'] == 1]
    color = VARIANT_COLORS[d['variant']]
    
    parts1 = ax.violinplot([neg_scores], positions=[pos], widths=0.9, showmeans=False, showmedians=True, showextrema=False)
    for body in parts1['bodies']:
        body.set_facecolor('#888'); body.set_alpha(0.55); body.set_edgecolor('#444')
    parts1['cmedians'].set_color('#222')
    
    parts2 = ax.violinplot([pos_scores], positions=[pos + 1], widths=0.9, showmeans=False, showmedians=True, showextrema=False)
    for body in parts2['bodies']:
        body.set_facecolor(color); body.set_alpha(0.75); body.set_edgecolor('#333')
    parts2['cmedians'].set_color('#222')
    
    positions.extend([pos, pos + 1])
    ticklabels.extend([f'{pretty_label(label)}\ny=0', f'{pretty_label(label)}\ny=1'])

ax.set_xticks(positions)
ax.set_xticklabels(ticklabels, fontsize=8)
ax.set_ylabel('Predicted probability of single-bidder')
ax.set_title('Predicted-score distributions by true class — wider separation = better model')
ax.set_ylim(-0.02, 1.02)
ax.axhline(0.5, color='#666', linestyle=':', linewidth=1, alpha=0.6)
savefig(fig, '13_score_distributions')

In [ ]:
# Single chart that uses the pooled model and walks the calendar month by month,
# computing AUC on each month's data. Reveals whether discriminability is stable
# through time or whether the regime boundary is a sharp break.

# Score every PLATINUM sample row with the pooled model.
X_score_all = X_pooled.copy()
score_all = models['pooled'].predict(X_score_all, num_iteration=models['pooled'].best_iteration)

scored = pd.DataFrame({
    'date': dates.values,
    'y_true': y.values,
    'y_score': score_all,
})
scored['month'] = pd.to_datetime(scored['date']).dt.to_period('M').dt.to_timestamp()

monthly_auc = []
for m, g in scored.groupby('month'):
    if g['y_true'].nunique() < 2 or len(g) < 200:
        continue
    auc = roc_auc_score(g['y_true'], g['y_score'])
    monthly_auc.append({'month': m, 'auc': auc, 'n': len(g), 'base_rate': g['y_true'].mean()})
monthly_auc = pd.DataFrame(monthly_auc, columns=['month', 'auc', 'n', 'base_rate'])

if len(monthly_auc) == 0:
    print('  [skip] no months met the minimum-sample threshold for rolling AUC')
else:
    fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True,
                             gridspec_kw={'height_ratios': [1.6, 1], 'hspace': 0.15})

    ax = axes[0]
    ax.plot(monthly_auc['month'], monthly_auc['auc'], color=VARIANT_COLORS['pooled'],
            linewidth=1.6, marker='o', markersize=3)
    ax.axvline(pd.Timestamp(REGIME_BOUNDARY), color='#d62728', linestyle='--', linewidth=1.3)
    ax.text(pd.Timestamp(REGIME_BOUNDARY), ax.get_ylim()[1] * 0.99,
            '  Resolution 1178', color='#d62728', fontsize=9, fontweight='bold', va='top')
    ax.set_ylabel('Monthly ROC-AUC')
    ax.set_title('Pooled model: monthly AUC and base rate over time')
    ax.set_ylim(max(0.5, monthly_auc['auc'].min() - 0.02), min(1.0, monthly_auc['auc'].max() + 0.02))

    ax = axes[1]
    ax.fill_between(monthly_auc['month'], monthly_auc['base_rate'],
                    color='#d62728', alpha=0.4, linewidth=0)
    ax.plot(monthly_auc['month'], monthly_auc['base_rate'], color='#7a0e0e', linewidth=1.4)
    ax.axvline(pd.Timestamp(REGIME_BOUNDARY), color='#d62728', linestyle='--', linewidth=1.3)
    ax.set_ylabel('Single-bidder rate')
    ax.set_xlabel('Month')

    savefig(fig, '14_monthly_rolling_auc')


In [ ]:
# A single composite figure: ROC overlay (top-left), PR overlay (top-right),
# transfer heatmap (bottom-left), top-10 SHAP comparison (bottom-right).
# This is the figure to drop into the paper's results section.

fig = plt.figure(figsize=(15, 12))
gs = GridSpec(2, 2, figure=fig, hspace=0.28, wspace=0.22)

# --- (a) ROC overlay ---
ax = fig.add_subplot(gs[0, 0])
for label, ls in order:
    if label not in all_predictions:
        continue
    d = all_predictions[label]
    fpr, tpr, _ = roc_curve(d['y_true'], d['y_score'])
    auc = roc_auc_score(d['y_true'], d['y_score'])
    color = VARIANT_COLORS[d['variant']]
    alpha = 1.0 if ls == 'solid' else 0.55
    ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=2 if ls == 'solid' else 1.4,
            alpha=alpha, label=f'{pretty_label(label)} ({auc:.3f})')
ax.plot([0, 1], [0, 1], color='#888', linestyle=':', linewidth=1)
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('(a) ROC — native and transfer'); ax.set_aspect('equal')
ax.legend(loc='lower right', fontsize=7.5)

# --- (b) PR overlay ---
ax = fig.add_subplot(gs[0, 1])
for label, ls in order:
    if label not in all_predictions:
        continue
    d = all_predictions[label]
    p, r, _ = precision_recall_curve(d['y_true'], d['y_score'])
    ap = average_precision_score(d['y_true'], d['y_score'])
    color = VARIANT_COLORS[d['variant']]
    alpha = 1.0 if ls == 'solid' else 0.55
    ax.plot(r, p, color=color, linestyle=ls, linewidth=2 if ls == 'solid' else 1.4,
            alpha=alpha, label=f'{pretty_label(label)} ({ap:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('(b) Precision-Recall'); ax.set_aspect('equal')
ax.legend(loc='lower left', fontsize=7.5)

# --- (c) Transfer heatmap ---
ax = fig.add_subplot(gs[1, 0])
im = ax.imshow(transfer_matrix.values.astype(float), cmap=cmap, vmin=0.5, vmax=1.0, aspect='auto')
for i in range(transfer_matrix.shape[0]):
    for j in range(transfer_matrix.shape[1]):
        v = transfer_matrix.values[i, j]
        if pd.notnull(v):
            ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                    color='white' if v > 0.78 else '#222', fontsize=12, fontweight='bold')
ax.set_xticks(range(transfer_matrix.shape[1])); ax.set_xticklabels(transfer_matrix.columns)
ax.set_yticks(range(transfer_matrix.shape[0])); ax.set_yticklabels(transfer_matrix.index)
ax.set_xlabel('Tested on'); ax.set_ylabel('Trained on')
ax.set_title('(c) Cross-regime transfer (AUC)'); ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.045, pad=0.04)

# --- (d) Top-10 SHAP comparison ---
ax = fig.add_subplot(gs[1, 1])
imp_wide_10 = imp_wide.head(10)
idx10 = np.arange(len(imp_wide_10))
bar_h = 0.27
offsets = {'peacetime': -bar_h, 'wartime': 0, 'pooled': bar_h}
for variant in imp_wide_10.columns:
    ax.barh(idx10 + offsets[variant], imp_wide_10[variant].values, height=bar_h,
            color=VARIANT_COLORS[variant], label=variant, alpha=0.92)
ax.set_yticks(idx10); ax.set_yticklabels(imp_wide_10.index, fontsize=8.5)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('(d) Top-10 SHAP feature importance')
ax.legend(loc='lower right', fontsize=8)

fig.suptitle(f'PLATINUM single-bidder model — headline results  ·  Run {RUN_ID}',
             y=0.995, fontsize=14, fontweight='bold')
savefig(fig, '15_summary_panel')

checkpoint(session, 'phase4_5_visualization', 'complete', {
    'figures_saved': sorted(p.name for p in FIG_DIR.glob('*.png')),
    'fig_dir': str(FIG_DIR),
})
print(f'\n{len(list(FIG_DIR.glob("*.png")))} figures saved to {FIG_DIR}')

---
## Phase 5: Persist Results, Upload Figures & Register Model

In [ ]:
session.sql('USE DATABASE PROZORRO').collect()
session.sql('USE SCHEMA PLATINUM').collect()

results_df['run_id'] = RUN_ID
results_df['regime_boundary'] = REGIME_BOUNDARY
results_df['n_features'] = X.shape[1]

results_snow = session.create_dataframe(results_df)
results_snow.write.save_as_table('PROZORRO.PLATINUM.R_PLATINUM_METRICS', mode='append')
print(f'Persisted {len(results_df)} metric rows to PROZORRO.PLATINUM.R_PLATINUM_METRICS')

for variant in ['peacetime', 'wartime', 'pooled']:
    imp = shap_results[variant]['importance'].copy()
    imp['variant'] = variant
    imp['run_id'] = RUN_ID
    imp['base_value'] = shap_results[variant]['base_value']
    imp_snow = session.create_dataframe(imp)
    imp_snow.write.save_as_table('PROZORRO.PLATINUM.R_PLATINUM_SHAP', mode='append')

print(f'Persisted SHAP importance to PROZORRO.PLATINUM.R_PLATINUM_SHAP')

checkpoint(session, 'phase5_persist', 'complete', {'run_id': RUN_ID})

In [ ]:
# Mirror all PNG + PDF figures to @PROZORRO.PLATINUM.PLATINUM_FIGURES/<RUN_ID>/
# so they're discoverable next to the metric tables.
session.sql('CREATE STAGE IF NOT EXISTS PROZORRO.PLATINUM.PLATINUM_FIGURES').collect()

stage_prefix = f'@PROZORRO.PLATINUM.PLATINUM_FIGURES/{RUN_ID}'
uploaded = []
for f in sorted(FIG_DIR.iterdir()):
    if f.suffix in {'.png', '.pdf'}:
        session.file.put(str(f), stage_prefix, auto_compress=False, overwrite=True)
        uploaded.append(f.name)
        print(f'  uploaded {f.name}')

checkpoint(session, 'phase5_figures', 'complete', {
    'stage_prefix': stage_prefix,
    'n_files': len(uploaded),
    'files': uploaded,
})
print(f'\n{len(uploaded)} files uploaded to {stage_prefix}')

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name='PROZORRO', schema_name='PLATINUM')

for variant in ['peacetime', 'wartime', 'pooled']:
    bst = models[variant]
    logs = training_logs[variant]
    model_name = f'PLATINUM_COMPETITIVENESS_{variant.upper()}'
    
    sample_input = pd.DataFrame(
        [[0] * datasets[variant]['train']['X'].shape[1]],
        columns=datasets[variant]['train']['X'].columns
    )
    for c in CATEGORICAL_FEATURES:
        if c in sample_input.columns:
            sample_input[c] = sample_input[c].astype('category')
    
    reg.log_model(
        model=bst,
        model_name=model_name,
        version_name=RUN_ID,
        metrics={
            'val_auc': logs['val_auc'],
            'val_logloss': logs['val_logloss'],
        },
        sample_input_data=sample_input,
        conda_dependencies=['lightgbm', 'scikit-learn', 'shap', 'matplotlib'],
        comment=f'Platinum pipeline {variant}. Regime boundary 2022-10-12. '
                f'Features: {logs["n_features"]}. Best iter: {logs["best_iteration"]}. '
                f'Figures @ {stage_prefix}.',
    )
    print(f'Registered: PROZORRO.PLATINUM.{model_name} @ {RUN_ID}')

checkpoint(session, 'phase5_registry', 'complete', {'run_id': RUN_ID})
print('\n=== PIPELINE COMPLETE ===')
print(f'Figures: {FIG_DIR}  (and {stage_prefix})')
print(f'Metrics: PROZORRO.PLATINUM.R_PLATINUM_METRICS  WHERE RUN_ID = \'{RUN_ID}\'')
print(f'SHAP:    PROZORRO.PLATINUM.R_PLATINUM_SHAP     WHERE RUN_ID = \'{RUN_ID}\'')

In [ ]:
import base64
from pathlib import Path

stage_path = f'@PROZORRO.PLATINUM.PLATINUM_FIGURES/{RUN_ID}/'
files_df = session.sql(f"LIST {stage_path}").to_pandas()
name_col = [c for c in files_df.columns if 'name' in c.lower()][0]

png_files = [row[name_col].split('/')[-1] for _, row in files_df.iterrows() if row[name_col].endswith('.png')]
print(f"Found {len(png_files)} PNG files")

local_tmp = Path('/tmp/fig_export')
local_tmp.mkdir(parents=True, exist_ok=True)

for fname in png_files:
    session.sql(f"GET {stage_path}{fname} file://{local_tmp}/").collect()

results = {}
for fname in png_files:
    fpath = local_tmp / fname
    if fpath.exists():
        with open(fpath, 'rb') as f:
            results[fname] = base64.b64encode(f.read()).decode('utf-8')

print(f"Encoded {len(results)} figures")
figure_data = results

In [ ]:
from pathlib import Path

tmp_dir = Path('/tmp/fig_dl2')

session.sql("CREATE STAGE IF NOT EXISTS PROZORRO.PLATINUM.WORKSPACE_EXPORT").collect()

for f in sorted(tmp_dir.glob('*.png')):
    session.sql(f"PUT file://{f} @PROZORRO.PLATINUM.WORKSPACE_EXPORT/prozorro-platinum/figures/ AUTO_COMPRESS=FALSE OVERWRITE=TRUE").collect()
    print(f"  uploaded {f.name}")

print("\nDone. Figures staged at @PROZORRO.PLATINUM.WORKSPACE_EXPORT/prozorro-platinum/figures/")